In [2]:
import scanpy as sc
import numpy as np
import squidpy as sq
import pandas as pd


/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [18]:
#sr_out = "/research/groups/plummgrp/projects/Visium_855118_Steve_Mack_Siri_09232025/common/Tuyu_Apr2026/Processed/846531_HD_BA_WM_C_DNB_SM_D90-7-D1-GFP_R1/outs/"
#sr_out = "/research/groups/plummgrp/projects/Visium_855118_Steve_Mack_Siri_09232025/common/Tuyu_Apr2026/Processed/846531_HD_BA_WM_C_DNB_SM_J74-3-A1-GFP_R1/outs/"
sr_out = "/research/groups/plummgrp/projects/Visium_855118_Steve_Mack_Siri_09232025/common/Tuyu_Apr2026/Processed/846531_HD_BA_WM_C_DNB_SM_J75-3-D1-GFP_R1/outs/"

adata = sq.read.visium(f"{sr_out}/binned_outputs/square_008um",counts_file="filtered_feature_bc_matrix.h5")

#label = "Tuyu_Apr2026_846531_HD_BA_WM_C_DNB_SM_D90_7_D1_GFP_R1_008um"
#label = "Tuyu_Apr2026_846531_HD_BA_WM_C_DNB_SM_J74_3_A1_GFP_R1_008um"
label = "Tuyu_Apr2026_846531_HD_BA_WM_C_DNB_SM_J75_3_D1_GFP_R1_008um"

#meta = sc.read_h5ad(f"Tuyu_Apr2026/spatial_{label}.h5ad")
meta = sc.read_h5ad(f"../data/processed/Tuyu_Apr2026/spatial_{label}.h5ad")

/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [19]:
common = adata.obs_names.intersection(meta.obs_names)

In [20]:
common

Index(['s_008um_00269_00526-1', 's_008um_00547_00611-1',
       's_008um_00260_00253-1', 's_008um_00433_00599-1',
       's_008um_00359_00037-1', 's_008um_00469_00254-1',
       's_008um_00435_00710-1', 's_008um_00188_00300-1',
       's_008um_00221_00585-1', 's_008um_00498_00370-1',
       ...
       's_008um_00358_00331-1', 's_008um_00399_00769-1',
       's_008um_00506_00476-1', 's_008um_00490_00635-1',
       's_008um_00366_00358-1', 's_008um_00398_00137-1',
       's_008um_00214_00470-1', 's_008um_00255_00356-1',
       's_008um_00565_00596-1', 's_008um_00414_00651-1'],
      dtype='object', length=216434)

In [21]:
adata = adata[common].copy()

/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [22]:
adata

AnnData object with n_obs × n_vars = 216434 × 19060
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

In [23]:
adata.obs["in_tissue"] = meta.obs.loc[common, "in_tissue"]

In [24]:
print("in_tissue:", adata.obs["in_tissue"].sum())

in_tissue: 216434


In [25]:
sc.pp.filter_cells(adata, min_counts=5)

/home/demerson/.conda/envs/spatial/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [26]:
print("in_tissue + >=5 UMIs:", adata.n_obs)

in_tissue + >=5 UMIs: 216434


In [28]:
tc = np.array(adata.X.sum(axis=1)).flatten()
for mc in [10, 20, 50, 100, 200, 500,750,1000,1300,1500]:
    print(f">= {mc:3d} UMIs: {(tc >= mc).sum():,}")

>=  10 UMIs: 216,434
>=  20 UMIs: 216,434
>=  50 UMIs: 216,434
>= 100 UMIs: 216,434
>= 200 UMIs: 216,434
>= 500 UMIs: 216,434
>= 750 UMIs: 185,480
>= 1000 UMIs: 149,332
>= 1300 UMIs: 105,290
>= 1500 UMIs: 80,224
